# Pipeline 159 — local Git-backed notebook

Inspect saved checkpoints and display them with the original evaluator output. The separate execution cell defaults to off and refuses to launch while the registered Docker runner has a fresh heartbeat. Defaults to Default → S1T; no special-case skip.

In [ ]:
from pathlib import Path
import contextlib
import fcntl
import pickle
import sys
import tempfile
import time
from types import SimpleNamespace

root = Path.home() / 'DataScience/Semester4/GA-Work/hybrid_variable_framework/Dynamic_Routing_Eval_Framework'
sys.path[:0] = [str(root), str(root / 'notebooks')]
from p159_staged_continuation import SharedStagingArchive, SharedCohortStore, WorkflowStore, optional_log
from p159_staged_continuation import GoogleDriveBackupManager, QubitAllocator, AllocatorRunner
from p159_staged_continuation import ExperimentConfiguration, MultiRunEvaluator
from tae_background_status import BackgroundWorkflowStatus

PIPELINE = 'Pipeline_159'
GIT_KEY = 'pipeline-159'
ALLOCATOR = 'Default'
COHORT = 'S1T'
RUN_MISSING_EXPERIMENTS = False

workflow_store = WorkflowStore()
_, index, workflow = workflow_store.load(PIPELINE)
settings = workflow['settings']
plan = workflow['plan']
if ALLOCATOR not in plan['allocators'] or COHORT not in plan['cohorts']:
    raise ValueError('Select a registered allocator and cohort')
replay = 'Tb' if COHORT.endswith('Tb') else 'T'
scale = float(COHORT[1:-len(replay)])
base_capacity = replay == 'Tb'
store = SharedCohortStore()
registered = store._read_index('index.json', store.ref('heads/main'))['pipelines'][GIT_KEY]
execution = registered['execution']
scenarios = {threat: threat for threat in settings['threats']}
models = settings['models']
runs = settings['runs']
base_frames, frame_step = settings['base_frames'], settings['frame_step']
scope = workflow['allocators'].get(ALLOCATOR, {}).get('cohorts', {}).get(COHORT, {})
pin = store.open_published(GIT_KEY, execution, COHORT) if COHORT in registered['cohorts'] else None
print(PIPELINE, execution, ALLOCATOR, COHORT, 'Git pin:', pin.head if pin else 'not published')


## Read and display completed experiments

This cell only reads Git. A missing checkpoint is not proof that the experiment never ran.

In [ ]:
def saved_runner(threat, position):
    key = f'{ALLOCATOR}/{threat}/{position}'
    if pin is None or not pin.experiments.get(key, {}).get('state'):
        return None
    state = pickle.loads(pin.read(key, 'state'))
    expected_frames = base_frames + (position - 1) * frame_step
    if state['id'] != position or not state['is_complete'] or state['frames_count'] != expected_frames or set(state['results']) != set(models):
        raise ValueError(f'Incompatible saved checkpoint: {key}')
    return dict(results=state['results'], winner=state.get('winner') or 'NA', exp_id=position)

saved = {threat: {} for threat in scenarios}
for threat in scenarios:
    for position in range(1, runs + 1):
        row = saved_runner(threat, position)
        if row is not None:
            saved[threat][position] = row
            frames = base_frames + (position - 1) * frame_step
            capacity = (base_frames if base_capacity else frames) * scale
            context = SimpleNamespace(frames_count=frames, configs=SimpleNamespace(
                allocator=ALLOCATOR, environment=row['results']['Oracle'].get('env_type', 'not recorded'),
                attack_strategy=threat, attack_rate=settings['attack_intensity'] if threat != 'none' else 0))
            print(f'⏩ SKIPPING EXPERIMENT {position}: ALREADY COMPLETED AND STORED')
            MultiRunEvaluator.display_run_results(context, position, saved[threat], capacity)
    recorded = scope.get('threats', {}).get(threat, {}).get('experiments', {})
    reported_only = [position for position in range(1, runs + 1)
                     if recorded.get(str(position), {}).get('state') == 'done' and position not in saved[threat]]
    print(f'{threat}: {len(saved[threat])}/{runs} verified from Git; recorded complete with no checkpoint: {reported_only}')


## Run the selected cohort locally (opt-in)

Only enable after the remote job has stopped and you have decided what to do about any historical results that were reported but not saved. For Default → S1T specifically, the old notebook skipped all 50 positions unconditionally; if no pin exists this cell refuses to guess whether they are new. Change no other pipeline.

In [ ]:
if RUN_MISSING_EXPERIMENTS:
    _, latest_index, latest_workflow = workflow_store.load(PIPELINE)
    heartbeat = latest_index[PIPELINE].get('runnerStatus') or {}
    if heartbeat.get('state') == 'running' and time.time() - heartbeat.get('updated', 0) < heartbeat.get('heartbeatTimeout', 180):
        raise RuntimeError('Pipeline 159 is already running. Do not start another copy.')
    if not pin and (ALLOCATOR, COHORT) == ('Default', 'S1T'):
        raise RuntimeError('Default/S1T was historically skipped and has no pinned evidence. Restore it or explicitly resolve its provenance before execution.')
    fresh = store.pipeline_cohorts(GIT_KEY, execution).get(COHORT)
    if fresh != registered['cohorts'].get(COHORT):
        raise RuntimeError('Git snapshot changed; refresh the read-only cells before execution.')
    if latest_workflow['settings'] != settings:
        raise RuntimeError('Registered settings changed; refresh before execution.')
    latest_scope = latest_workflow['allocators'].get(ALLOCATOR, {}).get('cohorts', {}).get(COHORT, {})
    binding = dict(pipelineId='159', driveId='0AE6N-7UYJfURUk9PVA')
    drive = GoogleDriveBackupManager.pipeline_creation_drive_service()
    lock_path = Path(tempfile.gettempdir()) / 'pipeline-159-local.lock'
    with lock_path.open('a') as lock:
        fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
        with BackgroundWorkflowStatus(workflow_store, PIPELINE, execution) as reporter:
            run = store.open_run(GIT_KEY, execution, COHORT)
            if run.experiments != (pin.experiments if pin else {}):
                raise RuntimeError('Unpublished runner records exist; inspect them before starting.')
            staging = Path(tempfile.mkdtemp(prefix='pipeline-159-local-staging-'))
            allocator = QubitAllocator() if ALLOCATOR == 'Default' else AllocatorRunner(
                ALLOCATOR, ['default'], {}, [scale], [runs], models, scenarios).create_allocator('default')
            config = ExperimentConfiguration(runs=runs, allocator=allocator, scenarios=scenarios, models=models,
                scale=scale, base_capacity=base_capacity, attack_intensity=settings['attack_intensity'],
                suffix=COHORT, resume=False, use_last_backup=False, overwrite=False, pipeline_storage=binding,
                staging_dir=staging, drive_service=drive)
            config.resume_allow_drive_downloads = False
            config.remote_experiment_archive = True
            evaluator = MultiRunEvaluator(configs=config, base_frames=base_frames, frame_step=frame_step,
                                          base_seed=settings['base_seed'])
            archive = SharedStagingArchive(config, run, staging)
            evaluator._experiment_archive = archive
            evaluator.env_experiments = saved
            reporter.start(ALLOCATOR, COHORT)
            for threat in scenarios:
                config.attack_type = threat
                recorded = latest_scope.get('threats', {}).get(threat, {}).get('experiments', {})
                for position in range(1, runs + 1):
                    if position in saved[threat]:
                        continue
                    if recorded.get(str(position), {}).get('state') == 'done':
                        print(f'{threat}/{position}: skipped completed experiment with missing checkpoint')
                        continue
                    with optional_log(tempfile.NamedTemporaryFile, mode='w', encoding='utf-8', dir=staging, prefix='experiment-log-') as log:
                        archive.log_path = Path(log.name) if log else None
                        archive.log_stream = log
                        from p159_staged_continuation import NotebookLog
                        with contextlib.redirect_stdout(NotebookLog(sys.stdout, log)):
                            evaluator.run_experiment(exp_no=position - 1)
            complete = all(set(saved[threat]) == set(range(1, runs + 1)) for threat in scenarios)
            if complete:
                evaluator.calculate_scenarios_performance()
            reporter.finish_cohort(complete=complete)
else:
    print('Read-only notebook check; no experiments started.')
